# 05_graphgrued_ordered5_prepare_windowed

Build a **leakage-safe early-prediction table** from the existing ordered-5 imputed table.

This notebook:
- drops legacy target columns from the old `is_completed` direction
- keeps only ordered-5 target columns
- creates 3 observation windows: **1 week, 2 weeks, 4 weeks**
- recomputes only **window-safe aggregates**
- saves one long table with `obs_window_weeks`

In [1]:

# CONFIG
INPUT_TABLE = "dbo.tbl_graphgrued_imputed_all_ordered5"
OUTPUT_TABLE = "dbo.tbl_graphgrued_ordered5_windowed"
WINDOWS = [1, 2, 4]

ID_COLS = ["user_id", "course_id", "enroll_time", "split_name", "imputer_name"]
TARGET_KEEP_COLS = [
    "risk_label_5_idx",
    "label_d_flag",
    "label_e_flag",
]
LEGACY_DROP_COLS = [
    "is_completed",
    "completion_rate",
    "risk_label_5_surrogate",
    "risk_label_5_name_a",
    "risk_label_5_name_b",
    "risk_label_5_name_c",
    "risk_label_5_name_d",
    "risk_label_5_name_e",
    "course_norm_score",
    "course_raw_score_sum",
    "problems_passed",
    "total_problems",
]


StatementMeta(, 15702bf0-eeb5-47bc-a3a3-0553111cbd04, 3, Finished, Available, Finished, False)

In [2]:

from pyspark.sql import functions as F

df = spark.table(INPUT_TABLE)
print("input rows:", df.count())
print("input cols:", len(df.columns))
display(df.limit(5))


StatementMeta(, 15702bf0-eeb5-47bc-a3a3-0553111cbd04, 4, Finished, Available, Finished, False)

input rows: 602454
input cols: 487


SynapseWidget(Synapse.DataFrame, 770c61b9-0e49-4160-9566-6bb1e4ddf7ae)

In [3]:

import pandas as pd
import numpy as np

pdf = df.toPandas()

# drop legacy / old-direction columns if present
for c in LEGACY_DROP_COLS:
    if c in pdf.columns:
        pdf = pdf.drop(columns=[c])

# keep target as integer 0..4
pdf["risk_label_5_idx"] = pd.to_numeric(pdf["risk_label_5_idx"], errors="coerce").astype("Int64")
pdf = pdf[pdf["risk_label_5_idx"].notna()].copy()
pdf["risk_label_5_idx"] = pdf["risk_label_5_idx"].astype(int)

bad = sorted(set(pdf["risk_label_5_idx"].unique().tolist()) - {0,1,2,3,4})
if bad:
    raise ValueError(f"Unexpected ordered-5 labels: {bad}")

weekly_bases = sorted(set(
    c.rsplit("_week", 1)[0]
    for c in pdf.columns
    if "_week" in c and c.rsplit("_week", 1)[1].isdigit()
))
print("weekly bases:", weekly_bases)
print("target values:", sorted(pdf["risk_label_5_idx"].unique().tolist()))


StatementMeta(, 15702bf0-eeb5-47bc-a3a3-0553111cbd04, 5, Finished, Available, Finished, False)

weekly bases: ['completed_questions', 'correct_answers', 'problem_attempt_count', 'rewind_count', 'rewind_distance', 'total_score', 'video_avg_chapter', 'video_count', 'video_watch_time']
target values: [0, 1, 2, 3, 4]


In [4]:

from typing import List

def cols_for_base(base: str, max_week: int) -> List[str]:
    cols = []
    for w in range(1, max_week + 1):
        c = f"{base}_week{w}"
        if c in pdf.columns:
            cols.append(c)
    return cols


def sum_cols(df_in: pd.DataFrame, cols: List[str]) -> pd.Series:
    if not cols:
        return pd.Series(np.zeros(len(df_in), dtype=np.float32), index=df_in.index)
    return df_in[cols].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1).astype(np.float32)


def mean_nonzero_cols(df_in: pd.DataFrame, cols: List[str]) -> pd.Series:
    if not cols:
        return pd.Series(np.zeros(len(df_in), dtype=np.float32), index=df_in.index)
    arr = df_in[cols].apply(pd.to_numeric, errors='coerce')
    return arr.mean(axis=1, skipna=True).fillna(0.0).astype(np.float32)


def build_window_df(df_in: pd.DataFrame, max_week: int) -> pd.DataFrame:
    keep_cols = [c for c in ID_COLS + TARGET_KEEP_COLS if c in df_in.columns]

    # keep only weekly cols up to max_week
    wk_cols = []
    for b in weekly_bases:
        wk_cols.extend(cols_for_base(b, max_week))

    out = df_in[keep_cols + wk_cols].copy()
    out["obs_window_weeks"] = int(max_week)
    out["obs_window_days"] = int(max_week * 7)

    # recompute safe aggregates only from weeks <= max_week
    out[f"video_count_total_w{max_week}"] = sum_cols(df_in, cols_for_base("video_count", max_week))
    out[f"video_watch_time_total_w{max_week}"] = sum_cols(df_in, cols_for_base("video_watch_time", max_week))
    out[f"rewind_count_total_w{max_week}"] = sum_cols(df_in, cols_for_base("rewind_count", max_week))
    out[f"rewind_distance_total_w{max_week}"] = sum_cols(df_in, cols_for_base("rewind_distance", max_week))
    out[f"problem_attempt_count_total_w{max_week}"] = sum_cols(df_in, cols_for_base("problem_attempt_count", max_week))
    out[f"total_score_total_w{max_week}"] = sum_cols(df_in, cols_for_base("total_score", max_week))
    out[f"correct_answers_total_w{max_week}"] = sum_cols(df_in, cols_for_base("correct_answers", max_week))
    out[f"completed_questions_total_w{max_week}"] = sum_cols(df_in, cols_for_base("completed_questions", max_week))

    ch_cols = cols_for_base("video_avg_chapter", max_week)
    out[f"avg_chapter_progress_w{max_week}"] = mean_nonzero_cols(df_in, ch_cols)

    if "course_total_videos" in df_in.columns:
        out["course_total_videos"] = pd.to_numeric(df_in["course_total_videos"], errors='coerce').fillna(0.0).astype(np.float32)
        denom = out["course_total_videos"].replace(0, np.nan)
        cov = out[f"video_count_total_w{max_week}"] / denom
        out[f"chapter_coverage_ratio_w{max_week}"] = cov.clip(lower=0.0, upper=1.0).fillna(0.0).astype(np.float32)

    if "course_max_chapter" in df_in.columns:
        out["course_max_chapter"] = pd.to_numeric(df_in["course_max_chapter"], errors='coerce').fillna(0.0).astype(np.float32)
        denom = out["course_max_chapter"].replace(0, np.nan)
        prog = out[f"avg_chapter_progress_w{max_week}"] / denom
        out[f"chapter_progress_ratio_w{max_week}"] = prog.clip(lower=0.0, upper=1.0).fillna(0.0).astype(np.float32)

    # active signal: how many observed weeks have any positive activity
    signal_cols = []
    for b in ["video_count", "problem_attempt_count", "rewind_count", "total_score"]:
        signal_cols.extend(cols_for_base(b, max_week))
    if signal_cols:
        sig = df_in[signal_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        week_active = []
        for w in range(1, max_week + 1):
            wcols = [c for c in signal_cols if c.endswith(f"week{w}")]
            if wcols:
                week_active.append((sig[wcols].sum(axis=1) > 0).astype(np.int32))
        if week_active:
            active_cnt = np.vstack([s.values for s in week_active]).T.sum(axis=1)
            out[f"activity_signal_w{max_week}"] = active_cnt.astype(np.float32)
            out[f"is_active_w{max_week}"] = (active_cnt > 0).astype(np.int32)

    # keep a few static safe columns if present
    safe_static = [
        "year_of_birth",
        "num_courses_enrolled",
        "num_fields",
        "num_resources",
        "num_active_enrollments",
        "course_total_videos",
        "course_max_chapter",
        "gender_0",
        "gender_1",
        "gender_2",
        "prerequisites",
    ]
    for c in safe_static:
        if c in df_in.columns and c not in out.columns:
            out[c] = df_in[c]

    return out

window_frames = []
for w in WINDOWS:
    tmp = build_window_df(pdf, w)
    window_frames.append(tmp)
    print(f"window {w}w shape:", tmp.shape)

window_pd = pd.concat(window_frames, axis=0, ignore_index=True)
print("windowed shape:", window_pd.shape)
print("cols:", len(window_pd.columns))
print(sorted(window_pd["obs_window_weeks"].unique().tolist()))


StatementMeta(, 15702bf0-eeb5-47bc-a3a3-0553111cbd04, 6, Finished, Available, Finished, False)

window 1w shape: (602454, 43)
window 2w shape: (602454, 52)
window 4w shape: (602454, 70)
windowed shape: (1807362, 96)
cols: 96
[1, 2, 4]


In [5]:

# final hygiene for Spark
import re
from collections import defaultdict

def spark_safe_unique_columns(cols):
    out = []
    seen = defaultdict(int)
    for c in cols:
        base = str(c).strip()
        base = re.sub(r"[^0-9a-zA-Z_]", "_", base)
        base = re.sub(r"_+", "_", base).strip("_").lower()
        if base == "":
            base = "col"
        k = seen[base]
        out.append(base if k == 0 else f"{base}__dup{k}")
        seen[base] += 1
    return out

window_pd.columns = spark_safe_unique_columns(window_pd.columns)
window_spark = spark.createDataFrame(window_pd)
spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_TABLE}")
window_spark.write.mode("overwrite").format("delta").saveAsTable(OUTPUT_TABLE)
print("saved:", OUTPUT_TABLE)
display(spark.table(OUTPUT_TABLE).limit(10))


StatementMeta(, 15702bf0-eeb5-47bc-a3a3-0553111cbd04, 7, Finished, Available, Finished, False)

saved: dbo.tbl_graphgrued_ordered5_windowed


SynapseWidget(Synapse.DataFrame, 820f8ff0-bcf4-48f9-a6bb-6ffdbcb4ef09)